### Building chatbot with memory
This notebook tells how a normal LLM conversation can become a chatbot that remembers previous messages,follow instructions,suppout different languages,can manage long conversations.



In [1]:
import os
from dotenv import load_dotenv
load_dotenv() #reads the dotenv file
groq_api_key=os.getenv("GROQ_API_KEY")
groq_api_key

'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'

The above cell helps to connect to the LLM.It retrieve the API key without show the secret key.

In [2]:
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGroq(
    model="openai/gpt-oss-20b",
    groq_api_key=groq_api_key
)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000019A06D239D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000019A06DF9050>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

here, a model object is created.It is now connected to the AI model.we can start conversation now.

In [3]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi,My name is Shivangi and I am chief AI engineer")])

AIMessage(content='Hello Shivangi! 👋 It’s great to meet the chief AI engineer. How can I assist you today? Whether you’re looking for insights, brainstorming ideas, or need help with a specific project, I’m here to help.', additional_kwargs={'reasoning_content': 'We need to respond. The user says "Hi,My name is Shivangi and I am chief AI engineer". We should respond politely, maybe ask how we can help. No policy violation.'}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 84, 'total_tokens': 180, 'completion_time': 0.097424115, 'completion_tokens_details': {'reasoning_tokens': 40}, 'prompt_time': 0.003940311, 'prompt_tokens_details': None, 'queue_time': 0.368803305, 'total_time': 0.101364426}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_bb6a356524', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1fedf-e52a-725b-a749-436285cbf7b6-0', tool_calls=[], invalid_tool_calls=[], us

HumanMessage represents the human message,AIMessage represents the AI message.invoke means execute the model with given input and give the response.

In [4]:
from langchain_core.messages import AIMessage 
model.invoke(
    [
        HumanMessage(content="Hi! My name is Shivangi and I am a AI Engineer."),
        AIMessage(content="Hello Shivangi! It's nice to meet you "),
        HumanMessage(content="Hey What's my name and what do I do")
    ]
)


AIMessage(content='You’re Shivangi, and you’re an AI Engineer.', additional_kwargs={'reasoning_content': 'User: "Hey What\'s my name and what do I do". The user wants to know their name and what they do. They have previously introduced themselves: "Hi! My name is Shivangi and I am a AI Engineer." So we should respond: "Your name is Shivangi, you are an AI Engineer." Possibly with some friendly tone. Ensure we don\'t mention policy.'}, response_metadata={'token_usage': {'completion_tokens': 98, 'prompt_tokens': 114, 'total_tokens': 212, 'completion_time': 0.102491531, 'completion_tokens_details': {'reasoning_tokens': 77}, 'prompt_time': 0.00588225, 'prompt_tokens_details': None, 'queue_time': 0.153562155, 'total_time': 0.108373781}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a8c584dda7', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1fedf-e8c1-76fa-b317-79744aa752db-0', tool_calls=[], invalid_tool_cal

But there is a problem:
You have to manually send the previous messages every single time.

This is not convenient for an actual chatbot. That's why the next topic is Message History.

Message History

We can use message history class to wrap our model and make it stateful.This will keep track of the inputs and outputs of the model,and store them in some datastore.Future interactions will then load those messages and pass them into the chain as part of the input.

In [5]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)


C:\Users\shiva\AppData\Local\Temp\ipykernel_25080\2013604763.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
c:\Users\shiva\anaconda3\envs\genai_env\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


store is an empty dictionary that will hold conversation histories, organized by session ID.
The wrapper(RunnableWithMessageHistory) takes responsibility for retrieving the history, sending it to the model and maintaining the conversation history for that session.

In [6]:
config={"configurable":{"session_id":"chat1"}}

In [7]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi,My name is Shivangi and I am chief AI engineer")],
    config=config
)

C:\Users\shiva\AppData\Local\Temp\ipykernel_25080\2013604763.py:9: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  store[session_id]=ChatMessageHistory()


In [8]:
response.content

'Hello Shivangi! 👋 It’s great to meet the chief AI engineer. How can I assist you today? Whether it’s brainstorming ideas, diving into code, or exploring the latest research, I’m here to help.'

In [9]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config
)

AIMessage(content='Your name is Shivangi.', additional_kwargs={'reasoning_content': 'User says "Hi, My name is Shivangi". So answer "Your name is Shivangi".'}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 143, 'total_tokens': 179, 'completion_time': 0.041438656, 'completion_tokens_details': {'reasoning_tokens': 21}, 'prompt_time': 0.069827559, 'prompt_tokens_details': None, 'queue_time': 0.31437648, 'total_time': 0.111266215}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_feb9b278f1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1fedf-ec08-769c-a02b-4d839456d329-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 143, 'output_tokens': 36, 'total_tokens': 179, 'output_token_details': {'reasoning': 21}})

Same session ID = same conversation memory.
If you change the ID to chat2, it becomes a different conversation.

In [10]:
##change the config->session_id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="What's my name")],
    config=config1
)

response.content

'I’m not sure what you’re called—could you let me know?'

In [11]:
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hey,My name is John")],
    config=config1
)

response.content

'Nice to meet you, John! How can I help you today?'

In [12]:
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="What's my name")],
    config=config1
)

response.content

'Your name is John.'

Prompt Template

Prompt Template helps to turn raw user information into a format that the LLM can work with.In this case,the raw user input is just a message,which we are passing to the LLM.Let's now make bit more complicated.First,let's add in a system message with some instructions(but still taking some messages as input).Next,we will add in more input besides just the messages.

A prompt template is used to control what instructions the model receives and how your input is organized.(in short)

In [13]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("You are a helpful assistant.Answer all the question to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)
chain=prompt|model

Component	Work  
system	: Instructions for how the AI should behave  
MessagesPlaceholder	: A placeholder where the messages/conversation will be inserted

our prompt template formats the raw input into the message structure that the LLM expects.
The prompt produces formatted messages, not the final answer.
Pipe passes that output to model
The model receives the formatted messages and generates an answer.

In [14]:
chain.invoke({"messages":[HumanMessage(content="My name is Shivangi")]})

AIMessage(content='Nice to meet you, Shivangi! How can I help you today?', additional_kwargs={'reasoning_content': 'User says: "My name is Shivangi". They likely want the assistant to respond acknowledging the name. Probably just respond with greeting. No other instructions. So answer: Hello Shivangi.'}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 96, 'total_tokens': 159, 'completion_time': 0.067447725, 'completion_tokens_details': {'reasoning_tokens': 39}, 'prompt_time': 0.006678673, 'prompt_tokens_details': None, 'queue_time': 0.256285305, 'total_time': 0.074126398}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e2cb7a84ec', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1fedf-f4bf-749b-9d03-482562c057aa-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 63, 'total_tokens': 159, 'output_token_details': {'reasoning': 39}})

In [15]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

c:\Users\shiva\anaconda3\envs\genai_env\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [16]:
config={"configurable":{"session_id":"chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi!My name is Shivangi")],
    config=config
)

In [17]:
response.content

'Hello, Shivangi! 👋 How can I help you today?'

In [18]:
##Add more complexity
prompt=ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant.Answer all the question to the best of your ability in {language}"
        ),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [19]:
response=chain.invoke({"messages":[HumanMessage(content="Hi!My name is Shivangi")],"language":"Hindi"})
response.content

'नमस्ते शिवांगी! आपका स्वागत है। मैं आपकी किस प्रकार सहायता कर सकती हूँ?'

Let's now wrap this more complicated chain in MessageHistory class.This time because there are multuple keys in the input,we need to specify the correct key to save the chat history.

In [20]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

c:\Users\shiva\anaconda3\envs\genai_env\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
config={"configurable":{"session_id":"chat4"}}
response=with_message_history.invoke(
    {'messages':[HumanMessage(content="Hi! I am Shivangi")],"language":"Hindi"},
    config=config
)

response.content

'नमस्ते शिवांगी जी! आपका स्वागत है। मैं आपकी किस प्रकार मदद कर सकता/सकती हूँ? यदि कोई सवाल है या किसी विषय पर जानकारी चाहिए, तो बताइए।'

In [24]:
response=with_message_history.invoke(
    {'messages':[HumanMessage(content="What's my name")],"language":"Hindi"},
    config=config
)

response.content

'आपका नाम शिवांगी है।'

Managing the conversation history

One inportant concept to uunderstand when building chatbots is how to manage conversation history.If left unmanged,the list of messages will grow unbounded and potentially overflow the context window of the LLM.Therefore,it is important to add a step to limits the size of messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

Imagine a chatbot that has been talking to a user for 500 messages. If all 500 messages are sent to the model every time, it consumes more tokens, increases costs/latency, and can exceed the model's context window.

In [28]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2+2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2+2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

Setting 	Meaning  
max_tokens=45	Keep history within the configured token limit  
strategy="last"	Prioritize the most recent messages  
token_counter=model	Use the model to estimate/count tokens  
include_system=True	Preserve system instructions if present  
allow_partial=False	Don't cut a message in the middle  
start_on="human"	Begin the retained sequence with a human message

In [29]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
chain=(RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
| prompt
| model
)

response=chain.invoke({
    "messages":messages+[HumanMessage(content="What ice cream do i like")],
    "language":"English"
}
)
response.content

'I’m not sure—could you tell me a bit about your favorite flavors or ingredients? That way I can suggest something you’ll probably love!'

itemgetter("messages") → Extracts messages from the input dictionary.  
| trimmer → Removes unnecessary old messages to fit the token limit.  
RunnablePassthrough.assign() → Keeps the original input and updates messages with trimmed messages.

In [30]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked: “What’s 2\u202f+\u202f2?”'

In [31]:
#Let's wrap this with message history
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

config={"configurable":{"session_id":"chat5"}}


c:\Users\shiva\anaconda3\envs\genai_env\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [32]:
response=chain.invoke({
    "messages":messages+[HumanMessage(content="What ice cream do i like")],
    "language":"English"
}
)
response.content

'I don’t have any record of your favorite ice‑cream flavors, but I’d love to help you figure it out! Here are a few ways we can narrow it down:\n\n| Category | Popular Choices | Why it’s a good pick |\n|----------|-----------------|----------------------|\n| Classic | Vanilla, chocolate, strawberry | Simple, timeless flavors |\n| Creamy | Cookies‑and‑cream, mint chocolate chip, salted caramel | Rich, indulgent textures |\n| Fruity | Lemon sorbet, mango, raspberry | Light, refreshing |\n| Nutty | Peanut butter cup, almond fudge, pistachio | Adds crunch and depth |\n| Seasonal | Pumpkin spice (fall), peppermint (winter), coconut‑lime (summer) | Tied to the time of year |\n| Experimental | Lavender honey, matcha, black sesame | Unique, adventurous |\n\n**Quick self‑check:**\n\n1. **Do you prefer creamy, dense textures or lighter, scoop‑and‑taste experiences?**  \n2. **Are you a fan of sweet vs. slightly salty or bitter notes?**  \n3. **Do you enjoy a nostalgic flavor (e.g., classic chocol